## 3. XGBoost Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_arrays, load_participant_ids
from src import metrics
from src import train
from src.models.classical_baseline import XGBoostPredictor, HORIZONS

MODEL_NAME = 'XGBoost'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'xgboost'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'xgboost_stability_summary.json'

In [3]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('train set shape: ', X_train.shape, '\nvaluation set shape: ', X_val.shape, '\ntest set shape: ', X_test.shape)
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

train set shape:  (83532, 24) 
valuation set shape:  (16739, 24) 
test set shape:  (16842, 24)

number of test participants:  6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"\nTest results — {MODEL_NAME} | seed={seed}\n")

    model = XGBoostPredictor(random_state=seed).fit(X_train, y_train)

    val_pred, test_pred = model.predict(X_val), model.predict(X_test)
    val_results = metrics.evaluate(val_pred, y_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(test_pred, y_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        test_pred, y_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'xgboost_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")


Test results — XGBoost | seed=7

 15 min | RMSE 11.83 | MAE 7.77 | gRMSE 13.38 | Zone A 96.92% | A+B 99.83%
 30 min | RMSE 19.08 | MAE 12.46 | gRMSE 22.37 | Zone A 90.17% | A+B 99.52%
 60 min | RMSE 27.59 | MAE 18.05 | gRMSE 33.89 | Zone A 81.59% | A+B 99.00%
 90 min | RMSE 32.63 | MAE 21.62 | gRMSE 40.83 | Zone A 75.69% | A+B 98.49%
120 min | RMSE 35.50 | MAE 23.74 | gRMSE 44.72 | Zone A 71.89% | A+B 98.03%

----------------------------------------------------------------------------------------------------

Test results — XGBoost | seed=14

 15 min | RMSE 11.86 | MAE 7.81 | gRMSE 13.42 | Zone A 96.90% | A+B 99.84%
 30 min | RMSE 19.08 | MAE 12.46 | gRMSE 22.36 | Zone A 90.19% | A+B 99.52%
 60 min | RMSE 27.58 | MAE 18.05 | gRMSE 33.89 | Zone A 81.61% | A+B 99.02%
 90 min | RMSE 32.68 | MAE 21.67 | gRMSE 40.90 | Zone A 75.82% | A+B 98.43%
120 min | RMSE 35.51 | MAE 23.79 | gRMSE 44.72 | Zone A 71.70% | A+B 98.02%

----------------------------------------------------------------------

In [5]:
# Test Results: overall mean +/- SD_seed (SD_participant)
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']
summary = {}

for h in HORIZONS:

    summary[h] = {}

    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []

        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }


print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

=== XGBoost test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.88 +/- 0.03 (2.50) mg/dL   MAE = 7.81 +/- 0.02 (1.76)   gRMSE = 13.44 +/- 0.04 (4.08)   Zone A = 96.89 +/- 0.04 (1.84)%   Zone A+B = 99.84 +/- 0.01 (0.22)%
 30-min:  RMSE = 19.11 +/- 0.03 (4.48) mg/dL   MAE = 12.47 +/- 0.02 (2.95)   gRMSE = 22.42 +/- 0.04 (7.41)   Zone A = 90.20 +/- 0.04 (3.35)%   Zone A+B = 99.53 +/- 0.01 (0.57)%
 60-min:  RMSE = 27.60 +/- 0.03 (9.42) mg/dL   MAE = 18.06 +/- 0.02 (5.54)   gRMSE = 33.90 +/- 0.04 (14.83)   Zone A = 81.65 +/- 0.04 (5.02)%   Zone A+B = 99.01 +/- 0.02 (0.91)%
 90-min:  RMSE = 32.65 +/- 0.02 (12.93) mg/dL   MAE = 21.65 +/- 0.02 (7.63)   gRMSE = 40.85 +/- 0.03 (20.21)   Zone A = 75.76 +/- 0.05 (6.72)%   Zone A+B = 98.47 +/- 0.02 (1.53)%
120-min:  RMSE = 35.52 +/- 0.03 (15.14) mg/dL   MAE = 23.78 +/- 0.02 (9.14)   gRMSE = 44.73 +/- 0.03 (23.51)   Zone A = 71.70 +/- 0.10 (7.21)%   Zone A+B = 98.03 +/- 0.02 (2.33)%

saved aggregate summary to /content/drive/

In [5]:
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)
print(f'\nsaved aggregate summary to {STABILITY_SUMMARY_PATH}')